# Predicción salarial: comparación de modelos tabulares

Este notebook usa la misma data, pero compara modelos clásicos de regresión para tabular y redes densas (MLP).
Incluye diagnóstico de por qué modelos secuenciales no se adaptan bien a esta estructura de datos.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)
SEED = 42
np.random.seed(SEED)

## 1) Carga de datos

In [3]:
DATA_PATH = 'tech_jobs_salaries.xlsx'

if DATA_PATH.lower().endswith('.xlsx'):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

display(df.head())
display(df.info())
display(df.isnull().sum().sort_values(ascending=False).head(15))

,job_title,company_size,employment_type,experience_level,years_experience,education_level,country,salary_local_currency,currency,remote_type,primary_skill,secondary_skill,work_hours_per_week,job_satisfaction_score,company_rating,age,gender
0,DevOps Engineer,SME,Full-time,Entry,3,Self-taught,India,1087489,INR,Onsite,SQL,Java,35,2.7,3.3,22,Female
1,Mobile App Developer,SME,Full-time,Lead,14,Bachelor,Japan,25198412,USD,Onsite,Docker,TensorFlow,57,2.8,3.9,51,Male
2,Cloud Engineer,Startup,Freelance,Senior,14,Self-taught,India,2579715,INR,Onsite,Kubernetes,C++,52,3.2,4.2,44,Female
3,Cybersecurity Analyst,Startup,Full-time,Entry,3,Diploma,India,949111,INR,Hybrid,NoSQL,Docker,40,3.4,3.3,38,Male
4,Backend Developer,SME,Contract,Lead,10,Self-taught,Japan,25195363,USD,Onsite,Linux,TensorFlow,56,2.8,4.5,54,Male


<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   job_title               200000 non-null  str    
 1   company_size            200000 non-null  str    
 2   employment_type         200000 non-null  str    
 3   experience_level        200000 non-null  str    
 4   years_experience        200000 non-null  int64  
 5   education_level         200000 non-null  str    
 6   country                 200000 non-null  str    
 7   salary_local_currency   200000 non-null  int64  
 8   currency                200000 non-null  str    
 9   remote_type             200000 non-null  str    
 10  primary_skill           200000 non-null  str    
 11  secondary_skill         200000 non-null  str    
 12  work_hours_per_week     200000 non-null  int64  
 13  job_satisfaction_score  200000 non-null  float64
 14  company_rating          200000 

None

job_title                 0
company_size              0
employment_type           0
experience_level          0
years_experience          0
education_level           0
country                   0
salary_local_currency     0
currency                  0
remote_type               0
primary_skill             0
secondary_skill           0
work_hours_per_week       0
job_satisfaction_score    0
company_rating            0
dtype: int64

## 2) Limpieza y target

- Normalizamos salarios a USD.
- Aplicamos `log1p` para estabilizar varianza.
- Recortamos extremos con IQR para reducir outliers extremos.

In [4]:
required_cols = ['salary_local_currency', 'currency']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Faltan columnas requeridas: {missing}')

currency_to_usd = {
    'USD': 1.0,
    'INR': 0.011,
    'EUR': 1.08,
    'GBP': 1.27,
    'CAD': 0.73,
    'AUD': 0.66,
    'MXN': 0.058,
    'COP': 0.00026,
}

work_df = df.copy()
work_df['currency_rate'] = work_df['currency'].map(currency_to_usd)
work_df['currency_rate'] = work_df['currency_rate'].fillna(1.0)

work_df['salary_usd'] = work_df['salary_local_currency'] * work_df['currency_rate']

q1 = work_df['salary_usd'].quantile(0.25)
q3 = work_df['salary_usd'].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

work_df['salary_usd_capped'] = work_df['salary_usd'].clip(lower, upper)
work_df['salary_log'] = np.log1p(work_df['salary_usd_capped'])

display(work_df['salary_log'].describe())

count    200000.000000
mean         11.572367
std           0.803348
min           8.034362
25%          11.209812
50%          11.651613
75%          12.094303
max          12.725901
Name: salary_log, dtype: float64

## 3) Ingeniería de variables

Creamos variables numéricas y categóricas a partir de niveles de experiencia, educación, tamaño de empresa y roles.

In [5]:
def safe_col(data, col, default=np.nan):
    return data[col] if col in data.columns else pd.Series(default, index=data.index)

exp_map = {
    'Entry': 0, 'Junior': 1, 'Mid': 2, 'Senior': 3, 'Lead': 4,
    'EN': 0, 'MI': 2, 'SE': 3, 'EX': 4
}
edu_map = {
    'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3,
    'Secondary': 0
}
remote_map = {
    'Onsite': 0, 'Hybrid': 1, 'Remote': 2,
    'On-site': 0
}
size_map = {
    'S': 0, 'M': 1, 'L': 2,
    'Small': 0, 'Medium': 1, 'Large': 2
}

exp = safe_col(work_df, 'experience_level').astype(str)
edu = safe_col(work_df, 'education_level').astype(str)
remote = safe_col(work_df, 'remote_type').astype(str)
size = safe_col(work_df, 'company_size').astype(str)
country = safe_col(work_df, 'country').astype(str)
job_title = safe_col(work_df, 'job_title').astype(str)
primary_skill = safe_col(work_df, 'primary_skill').astype(str)
secondary_skill = safe_col(work_df, 'secondary_skill').astype(str)

work_df['experience_level_num'] = exp.map(exp_map).fillna(2)
work_df['education_level_num'] = edu.map(edu_map).fillna(1)
work_df['remote_score'] = remote.map(remote_map).fillna(1)
work_df['company_size_num'] = size.map(size_map).fillna(1)

work_df['is_us'] = (country.str.upper() == 'UNITED STATES').astype(int)
work_df['same_skill_flag'] = (primary_skill.str.lower() == secondary_skill.str.lower()).astype(int)
work_df['job_title_len'] = job_title.str.len().fillna(job_title.str.len().median())

work_df['exp_x_edu'] = work_df['experience_level_num'] * work_df['education_level_num']
work_df['exp_x_remote'] = work_df['experience_level_num'] * work_df['remote_score']

high_demand_roles = ['data scientist', 'ml engineer', 'ai engineer', 'data engineer', 'software engineer']
work_df['high_demand_role'] = job_title.str.lower().apply(lambda x: int(any(r in x for r in high_demand_roles)))

## 4) Selección de variables y split

Evitamos leakage eliminando columnas directas del salario.

In [6]:
candidate_cat_cols = [
    'job_title', 'company_size', 'employment_type', 'experience_level', 'education_level',
    'country', 'remote_type', 'primary_skill', 'secondary_skill', 'gender'
]
cat_cols = [c for c in candidate_cat_cols if c in work_df.columns]

candidate_num_cols = [
    'experience_level_num', 'education_level_num', 'remote_score', 'company_size_num',
    'is_us', 'same_skill_flag', 'job_title_len', 'exp_x_edu', 'exp_x_remote', 'high_demand_role'
]
num_cols = [c for c in candidate_num_cols if c in work_df.columns]

drop_cols = [
    'salary_local_currency', 'salary_usd', 'salary_usd_capped', 'salary_log', 'currency', 'currency_rate'
]
drop_cols = [c for c in drop_cols if c in work_df.columns]

X = work_df.drop(columns=drop_cols, errors='ignore')
y = work_df['salary_log'].astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED
)

print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

KeyboardInterrupt: 

## 5) Preprocesamiento

- Numéricas: imputación de mediana + estandarización.
- Categóricas: imputación + one-hot.

In [ ]:
numeric_features = [c for c in num_cols if c in X.columns]
categorical_features = [c for c in cat_cols if c in X.columns]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print('Numéricas:', len(numeric_features), 'Categóricas:', len(categorical_features))

## 6) Modelos tabulares

Se compara contra un baseline y modelos no lineales que suelen funcionar mejor en tabular.

In [ ]:
def evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)

    def metrics(y_true, y_pred):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        return rmse, mae, r2

    y_pred_val = pipeline.predict(X_val)
    y_pred_test = pipeline.predict(X_test)

    rmse_val, mae_val, r2_val = metrics(y_val, y_pred_val)
    rmse_test, mae_test, r2_test = metrics(y_test, y_pred_test)

    y_test_usd = np.expm1(y_test)
    y_pred_test_usd = np.expm1(y_pred_test)
    rmse_usd = np.sqrt(mean_squared_error(y_test_usd, y_pred_test_usd))
    mae_usd = mean_absolute_error(y_test_usd, y_pred_test_usd)

    return {
        'RMSE_val_log': rmse_val,
        'MAE_val_log': mae_val,
        'R2_val_log': r2_val,
        'RMSE_test_log': rmse_test,
        'MAE_test_log': mae_test,
        'R2_test_log': r2_test,
        'RMSE_test_USD': rmse_usd,
        'MAE_test_USD': mae_usd,
        'pipeline': pipeline
    }

models = {
    'Baseline_Dummy': DummyRegressor(strategy='mean'),
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=SEED),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=SEED),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=SEED)
}

results = []
pipelines = {}
for name, model in models.items():
    print(f'Entrenando {name}...')
    out = evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test)
    pipelines[name] = out.pop('pipeline')
    out['Modelo'] = name
    results.append(out)

results_df = pd.DataFrame(results).sort_values('RMSE_test_log')
display(results_df)

## 7) MLP (red densa) para tabular

Una red densa (MLP) suele funcionar mejor que RNN/LSTM en tabular porque no impone un orden secuencial artificial.

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=200,
    random_state=SEED,
    early_stopping=True
)

print('Entrenando MLP...')
mlp_out = evaluate_model(mlp, X_train, X_val, X_test, y_train, y_val, y_test)
mlp_pipeline = mlp_out.pop('pipeline')
mlp_out['Modelo'] = 'MLPRegressor'
results_df = pd.concat([results_df, pd.DataFrame([mlp_out])], ignore_index=True)
results_df = results_df.sort_values('RMSE_test_log')
display(results_df)

## 8) Diagnóstico: por qué RNN/LSTM/GRU fallan en tabular

**Problemas típicos:**
- En tabular no hay un orden temporal real; forzar una secuencia degrada la señal.
- One-hot con miles de columnas crea secuencias largas con poca estructura.
- RNN/LSTM requieren más datos y regularización fuerte para generalizar.
- Sin embeddings entrenables para `job_title`/skills, se pierde semántica.
- El preprocesamiento con `StandardScaler` sobre one-hot puede ser subóptimo para redes recurrentes.

**Por qué modelos de árboles sí funcionan mejor:**
- Capturan interacciones no lineales en tabular con poca ingeniería.
- Soportan variables categóricas codificadas y outliers mejor.
- Generalizan mejor en datasets medianos.

## 9) Selección del mejor modelo y gráfico

Se toma el modelo con menor RMSE en test (log).

In [ ]:
best_model_name = results_df.iloc[0]['Modelo']
print('Mejor modelo:', best_model_name)

best_pipeline = pipelines.get(best_model_name, mlp_pipeline if best_model_name == 'MLPRegressor' else None)
if best_pipeline is None:
    raise ValueError('No se encontró pipeline para el mejor modelo.')

y_pred_best = best_pipeline.predict(X_test)
y_true_usd = np.expm1(y_test)
y_pred_usd = np.expm1(y_pred_best)

plt.figure(figsize=(6, 6))
plt.scatter(y_true_usd, y_pred_usd, alpha=0.2)
min_v = min(y_true_usd.min(), y_pred_usd.min())
max_v = max(y_true_usd.max(), y_pred_usd.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--')
plt.xlabel('Salario real USD')
plt.ylabel('Salario predicho USD')
plt.title(f'Real vs Predicho - {best_model_name}')
plt.show()

## 10) Próximos pasos

- Probar `CatBoostRegressor` o `XGBoost` si están disponibles.
- Usar embeddings para `job_title`/skills y un MLP más profundo.
- Incorporar variables temporales (año/mes) si existen en la data.
- Ajustar hiperparámetros con validación cruzada.